In [6]:
%matplotlib inline
import matplotlib.pyplot as plt

import healpy as hp
import numpy as np
import h5py, os

from astropy.io import fits
from functools import reduce

from msfm.utils import files
from msfm.utils.input_output import read_yaml

### global constants

In [7]:
conf = files.load_config( "../../../configs/marcel/simple.yaml")

n_side = conf["analysis"]["n_side"]
n_pix = conf["analysis"]["n_pix"]

pixels_file = f"../../../data/nside256/DESY3_pixels_{n_side}.h5"
noise_file = f"../../../data/nside256/DESY3_noise.h5"

# Metacal

In [8]:
# exclude the 0th bin, which includes all of the others
tomo_inds = [1, 2, 3, 4]
z_lims = conf["survey"]["lensing"]["z_lims"]
z_bins = conf["survey"]["lensing"]["z_bins"]

### load Metacal catalog

In [9]:
# data given by Dominik
data_dir = '/Users/arne/data/DESY3/DES_Y3KP_NGSF/'

e1_tomo = []
e2_tomo = []
w_tomo = []
n_bar_tomo = []
# consider all tomographic bins
for tomo in tomo_inds:
    # ellipticities for a fixed tomographic bin, these are doubles
    e1 = h5py.File(os.path.join(data_dir, f'cal_e1_tomo={tomo}.h5'))['cal_e1'][:]
    e2 = h5py.File(os.path.join(data_dir, f'cal_e2_tomo={tomo}.h5'))['cal_e2'][:]
    w = h5py.File(os.path.join(data_dir, f'weight_tomo={tomo}.h5'))['weight'][:]

    # J2000 angles in degrees
    alpha = h5py.File(os.path.join(data_dir, f'ALPHAWIN_J2000_tomo={tomo}.h5'))['ALPHAWIN_J2000'][:]
    delta = h5py.File(os.path.join(data_dir, f'DELTAWIN_J2000_tomo={tomo}.h5'))['DELTAWIN_J2000'][:]

    assert e1.shape == e2.shape == w.shape == alpha.shape == delta.shape

    # angles like in healpy in radian
    theta = -np.deg2rad(delta) + np.pi/2
    phi = np.deg2rad(alpha)

    # derived pixel ids, shape (num_galaxies,)
    indices = hp.ang2pix(nside=n_side, theta=theta, phi=phi)

    n_pix = len(np.unique(indices))
    n_gals = len(e1)

    n_bar = n_gals/n_pix

    print(f"tomo bin {tomo} contains {len(e1)} galaxies in {n_pix} pixels at n_side {n_side}")
    e1_tomo.append(e1)
    e2_tomo.append(e2)
    w_tomo.append(w)
    n_bar_tomo.append(n_bar)

tomo bin 1 contains 24940465 galaxies in 94573 pixels at n_side 256
tomo bin 2 contains 25280405 galaxies in 94570 pixels at n_side 256
tomo bin 3 contains 24891859 galaxies in 94568 pixels at n_side 256
tomo bin 4 contains 25091297 galaxies in 94540 pixels at n_side 256


# checks

In [10]:
# check that we have zero means
for tomo, e1, e2, w in zip(tomo_inds, e1_tomo, e2_tomo, w_tomo):
    # calculate
    print("Z bin: ", tomo)
    e1_mean = np.sum(e1*w)/np.sum(w)
    e2_mean = np.sum(e2*w)/np.sum(w)
    print("Mean e1: ", e1_mean)
    print("Mean e2: ", e2_mean)

Z bin:  1
Mean e1:  -1.0454848110573765e-18
Mean e2:  -2.2662481407338397e-18
Z bin:  2
Mean e1:  1.9298109539069204e-18
Mean e2:  4.4170040337157934e-18
Z bin:  3
Mean e1:  2.3564963418725306e-18
Mean e2:  -1.0293771723659463e-18
Z bin:  4
Mean e1:  1.65857831039232e-18
Mean e2:  8.000530525804698e-18


# create the noise file

In [11]:
with h5py.File(noise_file, "w") as f:
    for z_bin, e1, e2, w, n_bar in zip(z_bins, e1_tomo, e2_tomo, w_tomo, n_bar_tomo):
        # shape (n_gals, 3)
        sample = np.stack([e1, e2, w], axis=1)

        # save as np.float32, single precision
        dset = f.create_dataset(name=f"{z_bin}/cat", data=sample, dtype="f")
        dset.attrs["info"] = "This dataset contains the ellipticities of the Metacalibration galaxy sample per redshift bin. " \
                             "The shape is (n_gals, 3), where axis 1 contains (e1, e2, w)."

        dset = f.create_dataset(name=f"{z_bin}/n_bar", data=n_bar, dtype="f")

# testing

In [12]:
with h5py.File(noise_file, "r") as f:
    print(f.keys())
    print(f["metacal1"].keys())

    print(f["metacal1/n_bar"][()])
    print(f["metacal2/n_bar"][()])
    print(f["metacal3/n_bar"][()])
    print(f["metacal4/n_bar"][()])

<KeysViewHDF5 ['metacal1', 'metacal2', 'metacal3', 'metacal4']>
<KeysViewHDF5 ['cat', 'n_bar']>
263.71655
267.3195
263.21652
265.40402
